<a href="https://colab.research.google.com/github/EmanHrustemovic/FlyRank-AI-Intership-ML-Track-/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane 2: Refresh / Content Opportunity Scoring**

**Task type: Binary Classification + Ranking**

The core task is to classify each page as
"needs review" (positive) or "does not need
review" (negative), then rank all positive
candidates by priority score — so a reviewer
with limited capacity knows exactly where to
start.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy label):**
is_declining_label = (trend_direction == "down")

This is a proxy — not a perfect ground truth.
It captures pages currently showing a declining
trend signal, which is observable and safe to use.

A stronger future label would be:
- Features from prior 90 days → actual traffic
  decline in the next 30 days
But that requires the full warehouse (Week 3+),
so the proxy is the honest starting point.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50**

Why: A content reviewer can realistically check
~50 pages per week. Precision@50 asks: of the
top 50 pages the model recommends, how many
actually needed attention?

Starter baseline: Precision@50 = 0.240 (24%)
Starter random forest: Precision@50 = 0.740 (74%)

**Secondary metrics:**
- ROC AUC (overall ranking quality)
- Average Precision (whole ranking matters)
- Recall (missing a declining page is costly)

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd
import os

if not os.path.exists('/content/flyrank'):
    os.system('git clone https://github.com/EmanHrustemovic/FlyRank-AI-Intership-ML-Track- /content/flyrank')

df = pd.read_csv('/content/flyrank/data/raw/content_refresh_anonymized.csv')

# Define target
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Show unit of analysis
print("Unit of analysis: ONE ROW = ONE PAGE")
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['is_declining_label'].value_counts())
print(f"\nSample rows:")
df[['content_id', 'impressions_90d', 'ctr',
    'trend_direction', 'is_declining_label']].head(5)

Unit of analysis: ONE ROW = ONE PAGE
Shape: (30000, 45)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Sample rows:


,content_id,impressions_90d,ctr,trend_direction,is_declining_label
0,content_304f48230142,3803,0.76,down,1
1,content_a1fb4e703a9e,15320,0.05,down,1
2,content_9aa793d4d895,12581,0.09,down,1
3,content_331d6c4de07b,11751,0.49,stable,0
4,content_d99b7a2d90ca,19140,0.13,down,1


**What one row means:**
One row = one content page (pseudonymized).

Key observations from the sample:
- Shape: 30,000 pages × 45 columns
- Target is balanced enough: 16,262 declining (54.2%)
  vs 13,738 stable (45.8%)
- Example: content_a1fb4e703a9e has 15,320 impressions
  but only 0.05 CTR — high visibility, low engagement,
  clear review candidate
- Example: content_304f48230142 has 3,803 impressions
  and 0.76 CTR but is still declining — different
  problem, different action needed

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule like "flag all pages with
trend_direction == down" would flag 54.2%
of all pages — far too many for a reviewer
to handle.

ML beats a fixed rule because:
1. It combines multiple signals simultaneously
   (impressions, CTR, age, sessions, position)
   in ways a hand-written rule cannot capture
2. It learns which combinations actually matter
   from data — rather than guessing thresholds
3. It ranks candidates by probability, so
   the most urgent pages rise to the top
4. Starter results prove it: Precision@50
   jumps from 0.240 (rule) to 0.740 (model) —
   a 3x improvement in review efficiency

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.